In [1]:
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

import kagglehub


/kaggle/input/datasets/jvkrishwanth/extracted/df_main_phase1.csv
/kaggle/input/datasets/jvkrishwanth/extracted/extracted_dataset_ICA_CLEANED/P_27_MW-ica.fif
/kaggle/input/datasets/jvkrishwanth/extracted/extracted_dataset_ICA_CLEANED/P_14_MW-ica-metadata.mat
/kaggle/input/datasets/jvkrishwanth/extracted/extracted_dataset_ICA_CLEANED/P_3_MW-clean-epo.fif
/kaggle/input/datasets/jvkrishwanth/extracted/extracted_dataset_ICA_CLEANED/P_22_MW-clean-epo.fif
/kaggle/input/datasets/jvkrishwanth/extracted/extracted_dataset_ICA_CLEANED/P_15_MW-clean-epo.fif
/kaggle/input/datasets/jvkrishwanth/extracted/extracted_dataset_ICA_CLEANED/P_16_MW-ica-metadata.mat
/kaggle/input/datasets/jvkrishwanth/extracted/extracted_dataset_ICA_CLEANED/P_23_MW-ica.fif
/kaggle/input/datasets/jvkrishwanth/extracted/extracted_dataset_ICA_CLEANED/P_19_MW-clean-epo.fif
/kaggle/input/datasets/jvkrishwanth/extracted/extracted_dataset_ICA_CLEANED/P_7_MW-clean-epo.fif
/kaggle/input/datasets/jvkrishwanth/extracted/extracted_datas

In [2]:
"""Kaggle-ready raw-vs-ICA-cleaned EEG classification comparison.

Update DATA_ROOT if the Kaggle dataset is mounted somewhere else, then run:
    !pip install -q mne seaborn scikit-learn scipy
    !python kaggle_raw_vs_clean_extratrees.py

Expected layout under DATA_ROOT:
  Extracted/P_<id>_MW-epo.fif              # raw epochs
  ICA_cleaned/P_<id>_MW-clean-epo.fif      # ICA-cleaned epochs
  eeg_mw_results/final_exg_cleaned_all_phases/df_main_phase1.csv

The same hand-engineered features and the same epoch-grouped CV folds are used
for raw and cleaned data, so their metrics are directly comparable.
"""

from pathlib import Path
import re
import warnings

import mne
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.signal import butter, hilbert, sosfiltfilt, welch
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import StratifiedGroupKFold



RAW_DIR = Path("/kaggle/input/datasets/jvkrishwanth/extracted/extracted_dataset")
CLEAN_DIR = Path("/kaggle/input/datasets/jvkrishwanth/extracted/extracted_dataset_ICA_CLEANED")
LABEL_CSV = Path("/kaggle/input/datasets/jvkrishwanth/extracted/df_main_phase1.csv")
OUT_DIR = Path("/kaggle/working/raw_vs_clean_extratrees")

CHANNELS = [
    "Fp1", "Fp2", "F7", "F3", "Fz", "F4", "F8", "T3", "C3", "Cz", "C4",
    "T4", "T5", "P3", "Pz", "P4", "T6", "O1", "O2",
]
BANDS = {
    "Delta": (1, 4),
    "Theta": (4, 8),
    "Alpha": (8, 12),
    "Beta": (13, 30),
    "Gamma": (30, 45),
}
WINDOW_SECONDS = 2.0
STEP_SECONDS = 0.5
N_WINDOWS = 6
RANDOM_STATE = 42


def canonical_channel(name: str) -> str:
    """Make common FIF spellings match the canonical 19-channel montage."""
    aliases = {"FP1": "Fp1", "FP2": "Fp2", "PZ": "Pz", "CZ": "Cz", "FZ": "Fz"}
    return aliases.get(name.upper(), name)


def bandpass(data, low, high, sfreq):
    sos = butter(4, [low / (sfreq / 2), high / (sfreq / 2)], btype="band", output="sos")
    return sosfiltfilt(sos, data, axis=-1)


def burst_count(envelope):
    threshold = envelope.mean() + 2 * envelope.std()
    return float(np.sum(np.diff((envelope > threshold).astype(int)) == 1))


def subject_id(value) -> int:
    """Handles labels such as 1, 'P_1', 'sub-01', and '01'."""
    found = re.findall(r"\d+", str(value))
    if not found:
        raise ValueError(f"Cannot obtain a numeric subject id from {value!r}")
    return int(found[-1])


def subject_labels(df_main, sub_id):
    """Return labels indexed by each epoch's original source index."""
    sub = df_main[df_main["Subject"].map(subject_id) == sub_id]
    epochs = sub.groupby(["Session", "Epoch_Index"], sort=True)["State"].first().reset_index()
    return pd.Series(
        (epochs["State"].astype(str).str.upper() == "MW").astype(int).to_numpy(),
        index=epochs["Epoch_Index"].astype(int).to_numpy(),
    )


def load_epochs(fif_path):
    epochs = mne.read_epochs(fif_path, preload=True, verbose=False)
    rename = {name: canonical_channel(name) for name in epochs.ch_names}
    epochs.rename_channels(rename)
    missing = sorted(set(CHANNELS) - set(epochs.ch_names))
    if missing:
        raise ValueError(f"{fif_path.name} lacks required channels: {missing}")
    return epochs.copy().pick(CHANNELS)


def pair_raw_and_clean_epochs(raw_epochs, clean_epochs, labels_by_source_index, sub_id):
    """Pair on original events and obtain labels from their source indices."""
    raw_keys = [tuple(event[[0, 2]]) for event in raw_epochs.events]
    clean_keys = [tuple(event[[0, 2]]) for event in clean_epochs.events]
    clean_positions = {key: idx for idx, key in enumerate(clean_keys)}
    raw_idx = [idx for idx, key in enumerate(raw_keys) if key in clean_positions]
    clean_idx = [clean_positions[raw_keys[idx]] for idx in raw_idx]
    if not raw_idx:
        raise ValueError("no matching raw/clean event samples were found")
    raw_source_indices = np.asarray(raw_epochs.selection, dtype=int)[raw_idx]
    missing = sorted(set(raw_source_indices) - set(labels_by_source_index.index))
    if missing:
        raise ValueError(f"labels missing for raw source epoch indices: {missing}")
    paired_labels = labels_by_source_index.loc[raw_source_indices].to_numpy(dtype=int)
    print(f"Subject {sub_id}: using {len(raw_idx)} paired epochs (raw={len(raw_epochs)}, clean={len(clean_epochs)})")
    return raw_epochs[raw_idx], clean_epochs[clean_idx], paired_labels


def extract_features(epochs, labels, sub_id, condition):
    """Return six overlapping-window feature rows for every labeled epoch."""
    data = epochs.get_data(copy=True)
    if len(data) != len(labels):
        raise ValueError(
            f"Subject {sub_id}: {condition} has {len(data)} FIF epochs but {len(labels)} labels. "
            "This subject is skipped so labels are never misaligned."
        )

    sfreq = float(epochs.info["sfreq"])
    win_len, step = int(WINDOW_SECONDS * sfreq), int(STEP_SECONDS * sfreq)
    if data.shape[-1] < win_len + (N_WINDOWS - 1) * step:
        raise ValueError(f"{condition} epochs are too short for {N_WINDOWS} windows.")

    alpha_env = np.abs(hilbert(bandpass(data, 8, 12, sfreq), axis=-1))
    theta_env = np.abs(hilbert(bandpass(data, 4, 8, sfreq), axis=-1))
    rows = []

    for epoch_idx, label in enumerate(labels):
        for window_idx in range(N_WINDOWS):
            start, end = window_idx * step, window_idx * step + win_len
            raw = data[epoch_idx, :, start:end]
            a_env = alpha_env[epoch_idx, :, start:end]
            t_env = theta_env[epoch_idx, :, start:end]
            freqs, psd = welch(raw, fs=sfreq, nperseg=256, noverlap=128, axis=-1)
            total_mask = (freqs >= 1) & (freqs <= 45)
            alpha_corr = np.nan_to_num(np.corrcoef(a_env))

            row = {
                "Condition": condition,
                "Subject": sub_id,
                "Epoch_Index": epoch_idx,
                "Window_Index": window_idx,
                "Label": label,
            }
            for channel_idx, channel in enumerate(CHANNELS):
                channel_psd = psd[channel_idx]
                total = channel_psd[total_mask].mean()
                for band, (low, high) in BANDS.items():
                    mask = (freqs >= low) & (freqs < high if band != "Gamma" else freqs <= high)
                    absolute = channel_psd[mask].mean()
                    row[f"{channel}_{band}_Abs"] = absolute
                    row[f"{channel}_{band}_Rel"] = absolute / total if total > 0 else 0.0

                for prefix, env in (("Alpha", a_env[channel_idx]), ("Theta", t_env[channel_idx])):
                    mean = env.mean()
                    row[f"{channel}_{prefix}_Mean"] = mean
                    row[f"{channel}_{prefix}_Var"] = env.var()
                    row[f"{channel}_{prefix}_CV"] = env.std() / mean if mean > 0 else 0.0
                    row[f"{channel}_{prefix}_Bursts"] = burst_count(env)
                row[f"{channel}_Envelope_Sync"] = alpha_corr[channel_idx, np.arange(len(CHANNELS)) != channel_idx].mean()
            rows.append(row)
    return rows


def find_fif(folder, sub_id, cleaned):
    suffix = "-clean-epo.fif" if cleaned else "-epo.fif"
    exact = folder / f"P_{sub_id}_MW{suffix}"
    if exact.exists():
        return exact
    # Accommodates alternate capitalization/naming while still selecting the intended subject.
    candidates = [p for p in folder.rglob("*.fif") if re.search(rf"P_?{sub_id}(?!\d)", p.name, re.I)]
    candidates = [p for p in candidates if ("clean" in p.name.lower()) == cleaned and p.name.lower().endswith("-epo.fif")]
    return candidates[0] if len(candidates) == 1 else None


def make_dataset(df_main):
    rows = []
    for sub_id in sorted({subject_id(value) for value in df_main["Subject"].unique()}):
        raw_path = find_fif(RAW_DIR, sub_id, cleaned=False)
        clean_path = find_fif(CLEAN_DIR, sub_id, cleaned=True)
        if raw_path is None or clean_path is None:
            print(f"Skipping subject {sub_id}: missing raw or cleaned FIF.")
            continue
        try:
            labels = subject_labels(df_main, sub_id)
            raw_epochs, clean_epochs = load_epochs(raw_path), load_epochs(clean_path)
            raw_epochs, clean_epochs, paired_labels = pair_raw_and_clean_epochs(
                raw_epochs, clean_epochs, labels, sub_id
            )
            rows.extend(extract_features(raw_epochs, paired_labels, sub_id, "Raw"))
            rows.extend(extract_features(clean_epochs, paired_labels, sub_id, "ICA-cleaned"))
            print(f"Finished subject {sub_id}")
        except ValueError as exc:
            warnings.warn(f"Skipping subject {sub_id}: {exc}")
    return pd.DataFrame(rows)


def evaluate_condition(df, condition):
    subset = df[df["Condition"] == condition].reset_index(drop=True)
    metadata = {"Condition", "Subject", "Epoch_Index", "Window_Index", "Label"}
    features = [column for column in subset.columns if column not in metadata]
    X = np.nan_to_num(subset[features].to_numpy(), nan=0.0, posinf=0.0, neginf=0.0)
    y = subset["Label"].to_numpy()
    # All overlapping windows from a trial stay together in exactly one fold.
    groups = subset["Subject"].astype(str) + "_" + subset["Epoch_Index"].astype(str)
    cv = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
    predictions, probabilities = np.zeros(len(y), dtype=int), np.zeros(len(y))

    for train_idx, test_idx in cv.split(X, y, groups):
        model = ExtraTreesClassifier(
            n_estimators=300, class_weight="balanced", random_state=RANDOM_STATE, n_jobs=-1
        )
        model.fit(X[train_idx], y[train_idx])
        predictions[test_idx] = model.predict(X[test_idx])
        probabilities[test_idx] = model.predict_proba(X[test_idx])[:, 1]

    # Train once on all data only for a stable feature-importance summary, not performance reporting.
    importance_model = ExtraTreesClassifier(
        n_estimators=300, class_weight="balanced", random_state=RANDOM_STATE, n_jobs=-1
    ).fit(X, y)
    importance = pd.DataFrame({"Feature": features, "Importance": importance_model.feature_importances_})
    importance = importance.sort_values("Importance", ascending=False)
    metrics = {
        "Condition": condition,
        "Accuracy": accuracy_score(y, predictions),
        "Precision": precision_score(y, predictions, zero_division=0),
        "Recall": recall_score(y, predictions, zero_division=0),
        "F1": f1_score(y, predictions, zero_division=0),
        "ROC-AUC": roc_auc_score(y, probabilities),
    }
    return metrics, importance, y, predictions


def plot_outputs(metrics_df, results):
    sns.set_theme(style="whitegrid")
    metric_columns = ["Accuracy", "Precision", "Recall", "F1", "ROC-AUC"]
    long_metrics = metrics_df.melt(id_vars="Condition", value_vars=metric_columns, var_name="Metric", value_name="Score")
    plt.figure(figsize=(10, 6))
    ax = sns.barplot(data=long_metrics, x="Metric", y="Score", hue="Condition", palette="Set2")
    ax.set_ylim(0, 1)
    ax.set_title("Extra Trees: raw vs ICA-cleaned EEG classification")
    for container in ax.containers:
        ax.bar_label(container, fmt="%.3f", padding=2, fontsize=8)
    plt.tight_layout()
    plt.savefig(OUT_DIR / "raw_vs_clean_classification_metrics.png", dpi=200)
    plt.close()

    for condition, (_, _, y_true, y_pred) in results.items():
        cm = confusion_matrix(y_true, y_pred)
        plt.figure(figsize=(5, 4))
        sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", cbar=False,
                    xticklabels=["Focused", "MW"], yticklabels=["Focused", "MW"])
        plt.title(f"{condition}: out-of-fold confusion matrix")
        plt.xlabel("Predicted")
        plt.ylabel("True")
        plt.tight_layout()
        plt.savefig(OUT_DIR / f"{condition.lower().replace('-', '_')}_confusion_matrix.png", dpi=200)
        plt.close()

    importance = results["ICA-cleaned"][1]
    def feature_band(feature):
        if "Envelope_Sync" in feature:
            return "Alpha"
        return next((band for band in BANDS if f"_{band}_" in feature), None)
    band_importance = importance.assign(Band=importance["Feature"].map(feature_band))
    band_importance = band_importance.dropna(subset=["Band"])
    band_importance = band_importance.groupby("Band", as_index=False)["Importance"].sum()
    band_importance["Importance"] = band_importance["Importance"] / band_importance["Importance"].sum()
    band_importance = band_importance.sort_values("Importance", ascending=False)
    band_importance.to_csv(OUT_DIR / "bandwise_feature_importance.csv", index=False)
    plt.figure(figsize=(8, 5))
    sns.barplot(data=band_importance, x="Band", y="Importance", order=band_importance["Band"], palette="viridis")
    plt.title("Total Extra Trees Feature Importance by Band")
    plt.tight_layout()
    plt.savefig(OUT_DIR / "bandwise_feature_importance.png", dpi=200)
    plt.close()

def main():
    OUT_DIR.mkdir(parents=True, exist_ok=True)
    label_csv = Path(LABEL_CSV)
    if not label_csv.exists():
        raise FileNotFoundError(f"Set LABEL_CSV correctly. Not found: {label_csv}")
    df_main = pd.read_csv(label_csv)
    required = {"Subject", "Session", "Epoch_Index", "State"}
    missing = required - set(df_main.columns)
    if missing:
        raise ValueError(f"df_main_phase1.csv is missing columns: {sorted(missing)}")

    dataset = make_dataset(df_main)
    if dataset.empty:
        raise RuntimeError("No paired raw/clean subjects were processed. Check the three input paths.")
    dataset.to_csv(OUT_DIR / "raw_and_clean_windowed_features.csv", index=False)

    results = {}
    for condition in ("Raw", "ICA-cleaned"):
        metrics, importance, y_true, y_pred = evaluate_condition(dataset, condition)
        results[condition] = (metrics, importance, y_true, y_pred)
        importance.to_csv(OUT_DIR / f"{condition.lower().replace('-', '_')}_feature_importance.csv", index=False)

    metrics_df = pd.DataFrame([result[0] for result in results.values()])
    metrics_df.to_csv(OUT_DIR / "raw_vs_clean_metrics.csv", index=False)
    plot_outputs(metrics_df, results)
    print("\nCompleted. Outputs written to:", OUT_DIR)
    print(metrics_df.to_string(index=False))


if __name__ == "__main__":
    main()


Subject 1: using 40 paired epochs (raw=40, clean=40)
Finished subject 1
Subject 2: using 40 paired epochs (raw=40, clean=40)
Finished subject 2
Subject 3: using 40 paired epochs (raw=40, clean=40)
Finished subject 3
Subject 5: using 40 paired epochs (raw=40, clean=40)
Finished subject 5
Subject 7: using 40 paired epochs (raw=40, clean=40)
Finished subject 7
Subject 8: using 40 paired epochs (raw=40, clean=40)
Finished subject 8
Subject 9: using 40 paired epochs (raw=40, clean=40)
Finished subject 9
Subject 10: using 40 paired epochs (raw=40, clean=40)
Finished subject 10
Subject 11: using 40 paired epochs (raw=40, clean=40)
Finished subject 11
Subject 12: using 40 paired epochs (raw=40, clean=40)
Finished subject 12
Subject 13: using 40 paired epochs (raw=40, clean=40)
Finished subject 13
Subject 14: using 40 paired epochs (raw=40, clean=40)
Finished subject 14
Subject 15: using 40 paired epochs (raw=40, clean=40)
Finished subject 15
Subject 16: using 40 paired epochs (raw=40, clean=40

/tmp/ipykernel_58/2320420090.py:286: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(data=band_importance, x="Band", y="Importance", order=band_importance["Band"], palette="viridis")



Completed. Outputs written to: /kaggle/working/raw_vs_clean_extratrees
  Condition  Accuracy  Precision   Recall       F1  ROC-AUC
        Raw  0.689833   0.715602 0.476245 0.571889 0.723432
ICA-cleaned  0.689167   0.716696 0.472031 0.569185 0.730473
